# 14c -- Stage 2 Multi-Crop TTA

Kernel slug: `yahiaakhalafallah/14c-tta-stage2`.

Variant: **14c v1: hflip + scale TTA**.

This experiment reuses the proven 14a stages 2-5 notebook shape and the same 10a Stage 0/1 checkpoint, but runs vanilla Stage 2 ReID extraction with test-time augmentation only. It does not install, download, or enable SAM2 masking. Internal frame IDs remain the 0-based Stage 1 convention; MOT output is converted to 1-based in Stage 5.

In [ ]:
import os, sys, subprocess, shutil, json, time, tarfile, re
from pathlib import Path

# Guard: detect GPU before importing torch. Kaggle's newest PyTorch builds may
# not support P100 sm_60, so pin the cu124 build requested for this experiment.
if shutil.which("nvidia-smi"):
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=gpu_name,compute_cap", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    if result.returncode == 0 and result.stdout.strip():
        gpu_name, compute_cap = result.stdout.strip().split(",", 1)
        match = re.search(r"(\d+)\.(\d+)", compute_cap)
        if match:
            major, minor = match.groups()
            sm = int(major) * 10 + int(minor)
            if sm < 70:
                print(f"GPU {gpu_name.strip()} sm_{sm}: installing torch 2.4.1+cu124")
                subprocess.check_call([
                    sys.executable, "-m", "pip", "install", "-q",
                    "torch==2.4.1+cu124", "torchvision==0.19.1+cu124",
                    "--index-url", "https://download.pytorch.org/whl/cu124",
                ])

import torch

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"  GPU {index}: {torch.cuda.get_device_name(index)} ({props.total_memory / 1024**3:.1f} GB)")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

## 1. Clone Repo And Install Dependencies

In [ ]:
REPO_URL = "https://github.com/MRKDaGods/gp.git"
WORK_DIR = Path("/kaggle/working")
PROJECT = WORK_DIR / "gp"

if not PROJECT.exists():
    print(f"Cloning {REPO_URL} ...")
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", "feature/pretrained-ensemble", REPO_URL, str(PROJECT)])
else:
    print("Repo already present; pulling latest ...")
    subprocess.check_call(["git", "-C", str(PROJECT), "pull", "--ff-only"])

os.chdir(str(PROJECT))
sys.path.insert(0, str(PROJECT))
print(f"Repo ready at {PROJECT}")

In [ ]:
def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# Keep torch/torchvision pinned above; install project dependencies around it.
pip("--no-deps", "ultralytics")
pip("filterpy", "ftfy", "lapx")
pip("--no-deps", "boxmot==11.0.3")

try:
    import torchreid
    print("torchreid ok")
except ImportError:
    pip("git+https://github.com/KaiyangZhou/deep-person-reid.git")

try:
    import faiss
    print(f"faiss ok ({faiss.__version__})")
except ImportError:
    try:
        pip("faiss-gpu")
    except Exception:
        pip("faiss-cpu")

pip("timm", "motmetrics")
pip("loguru", "omegaconf", "rich", "networkx>=3.1", "click")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], cwd=str(PROJECT))
print("Dependencies installed")

In [ ]:
FAILED = []
for label, module in [
    ("ultralytics", "ultralytics"),
    ("boxmot", "boxmot"),
    ("torch", "torch"),
    ("torchreid", "torchreid"),
    ("timm", "timm"),
    ("faiss", "faiss"),
    ("motmetrics", "motmetrics"),
    ("cv2", "cv2"),
    ("omegaconf", "omegaconf"),
]:
    try:
        __import__(module)
        print(f"  OK {label}")
    except ImportError as exc:
        print(f"  MISSING {label}: {exc}")
        FAILED.append(label)
if FAILED:
    raise RuntimeError(f"Missing modules: {FAILED}")
print("All required modules importable")

## 2. Mount MTMC Weights

In [ ]:
import zipfile

weights_candidates = [
    Path("/kaggle/input/mtmc-weights"),
    Path("/kaggle/input/yahiaakhalafallah-mtmc-weights"),
    Path("/kaggle/input/yahiaakhalafallah/mtmc-weights"),
    Path("/kaggle/input/gumfreddy-mtmc-weights"),
    Path("/kaggle/input/gumfreddy/mtmc-weights"),
]
WEIGHTS_INPUT = next((path for path in weights_candidates if path.exists()), None)
if WEIGHTS_INPUT is None:
    candidates = [
        path
        for path in Path("/kaggle/input").rglob("*")
        if path.is_dir() and "mtmc" in path.name.lower() and "weight" in path.name.lower()
    ]
    if candidates:
        WEIGHTS_INPUT = candidates[0]
    else:
        tried = ", ".join(str(path) for path in weights_candidates)
        raise FileNotFoundError(
            f"MTMC weights dataset not found; attach yahiaakhalafallah/mtmc-weights in kernel-metadata.json. Tried: {tried}"
        )
print(f"MTMC weights input: {WEIGHTS_INPUT}")

MODELS_DST = PROJECT / "models"
if MODELS_DST.is_symlink():
    MODELS_DST.unlink()
if MODELS_DST.exists():
    shutil.rmtree(MODELS_DST)
print(f"Copying models from {WEIGHTS_INPUT} ...")
shutil.copytree(str(WEIGHTS_INPUT), str(MODELS_DST))

for zip_path in sorted(MODELS_DST.rglob("*.zip")):
    print(f"Extracting {zip_path.relative_to(MODELS_DST)}")
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(zip_path.parent)
    zip_path.unlink()

for subdir in ["detection", "reid", "tracker"]:
    (MODELS_DST / subdir).mkdir(exist_ok=True)
for path in list(MODELS_DST.glob("*.pt")):
    if "yolo" in path.name.lower():
        shutil.move(str(path), str(MODELS_DST / "detection" / path.name))
    elif "osnet" in path.name.lower():
        shutil.move(str(path), str(MODELS_DST / "tracker" / path.name))
for path in list(MODELS_DST.glob("*.pth")):
    shutil.move(str(path), str(MODELS_DST / "reid" / path.name))
for path in list(MODELS_DST.glob("*.pkl")):
    shutil.move(str(path), str(MODELS_DST / "reid" / path.name))
for path in list(MODELS_DST.glob("*.json")):
    if path.name != "dataset-metadata.json":
        shutil.move(str(path), str(MODELS_DST / "reid" / path.name))

essential = [
    PROJECT / "models" / "reid" / "transreid_cityflowv2_best.pth",
]
missing = [str(path) for path in essential if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing essential weights: {missing}")
print("Baseline ReID weights ready")

In [ ]:
# 14c v1 TTA patch: explicit per-model view lists with mean_l2 aggregation.
os.environ["MTMC_TTA_RECIPE"] = "14c_v1"

TTA_RECIPE = {
    "variant": "14c v1: hflip + scale TTA",
    "primary_views": ["original", "hflip", "scale_0.95", "scale_1.05"],
    "tertiary_dinov2_views": ["original", "hflip"],
    "aggregation": "mean_l2",
}

REID_MODEL_PATH = PROJECT / "src" / "stage2_features" / "reid_model.py"
reid_text = REID_MODEL_PATH.read_text(encoding="utf-8")


def replace_once(text: str, old: str, new: str, label: str) -> str:
    if old not in text:
        raise RuntimeError(f"14c TTA patch anchor not found: {label}")
    return text.replace(old, new, 1)

if "MTMC_TTA_RECIPE" not in reid_text:
    reid_text = replace_once(
        reid_text,
        "from typing import List, Optional, Tuple, TYPE_CHECKING\n\nimport cv2",
        "from typing import List, Optional, Tuple, TYPE_CHECKING\n\nimport os\n\nimport cv2",
        "import os",
    )
    reid_text = replace_once(
        reid_text,
        "        self.vit_model = vit_model\n\n        # Auto-detect CLIP normalization",
        """        self.vit_model = vit_model
        self.tta_views: list[str] = []
        tta_recipe = os.environ.get(\"MTMC_TTA_RECIPE\", \"\")
        if tta_recipe == \"14c_v1\" and self.is_transreid:
            if \"dinov2\" in vit_model.lower():
                self.tta_views = [\"original\", \"hflip\"]
            else:
                self.tta_views = [\"original\", \"hflip\", \"scale_0.95\", \"scale_1.05\"]
            self.normalize_views = True

        # Auto-detect CLIP normalization""",
        "init tta views",
    )
    reid_text = replace_once(
        reid_text,
        "            + (\", normalize_views=true\" if self.normalize_views else \"\")\n        )",
        "            + (\", normalize_views=true\" if self.normalize_views else \"\")\n            + (f\", tta_views={self.tta_views}\" if self.tta_views else \"\")\n        )",
        "log tta views",
    )
    reid_text = replace_once(
        reid_text,
        "    @torch.no_grad()\n    def _extract_batch",
        """    def _apply_tta_view(self, crops: List[np.ndarray], view_name: str) -> List[np.ndarray]:
        if view_name == \"original\":
            return crops
        if view_name == \"hflip\":
            return [cv2.flip(crop, 1) for crop in crops]
        if view_name.startswith(\"scale_\"):
            return self._center_crop_at_scale(crops, float(view_name.split(\"_\", 1)[1]))
        raise ValueError(f\"Unsupported TTA view: {view_name}\")

    @torch.no_grad()
    def _extract_batch""",
        "apply_tta_view helper",
    )
    reid_text = replace_once(
        reid_text,
        "        cam_tensor = self._make_cam_tensor(len(batch_crops), cam_id)\n        views = [self._forward_crops(batch_crops, cam_tensor)]",
        """        cam_tensor = self._make_cam_tensor(len(batch_crops), cam_id)
        if self.tta_views:
            views = [
                self._forward_crops(self._apply_tta_view(batch_crops, view_name), cam_tensor)
                for view_name in self.tta_views
            ]
            views = [self._l2_normalize_rows(view) for view in views]
            pooled = np.mean(np.stack(views, axis=0), axis=0)
            return self._l2_normalize_rows(pooled).astype(np.float32)

        views = [self._forward_crops(batch_crops, cam_tensor)]""",
        "extract_batch exact views",
    )
    REID_MODEL_PATH.write_text(reid_text, encoding="utf-8")

print("Applied 14c TTA patch")
print(json.dumps(TTA_RECIPE, indent=2))

## 3. Mount CityFlowV2 Videos

In [ ]:
import re as regex

for mount in ["/tmp", "/kaggle/working"]:
    total, used, free = shutil.disk_usage(mount)
    print(f"{mount:16s} {free / 1024**3:.1f} GB free / {total / 1024**3:.1f} GB total")

candidate_mounts = [
    Path("/kaggle/input/data-aicity-2023-track-2"),
    Path("/kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2"),
]
CITYFLOW_INPUT = next((path for path in candidate_mounts if path.exists()), None)
if CITYFLOW_INPUT is None:
    raise FileNotFoundError("CityFlowV2 dataset not found; attach thanhnguyenle/data-aicity-2023-track-2")
print(f"CityFlowV2 input: {CITYFLOW_INPUT}")

TMP_DATA = Path("/tmp/datasets")
TMP_DATA.mkdir(parents=True, exist_ok=True)
DATA_RAW_PARENT = PROJECT / "data" / "raw"
if not DATA_RAW_PARENT.is_symlink():
    if DATA_RAW_PARENT.exists():
        shutil.rmtree(DATA_RAW_PARENT)
    DATA_RAW_PARENT.parent.mkdir(parents=True, exist_ok=True)
    DATA_RAW_PARENT.symlink_to(TMP_DATA)

DATA_RAW = TMP_DATA / "cityflowv2"
DATA_RAW.mkdir(parents=True, exist_ok=True)
for split_dir in sorted(CITYFLOW_INPUT.iterdir()):
    if not split_dir.is_dir() or split_dir.name not in ("train", "validation", "test"):
        continue
    for scene_dir in sorted(split_dir.iterdir()):
        if not scene_dir.is_dir():
            continue
        for cam_dir in sorted(scene_dir.iterdir()):
            if not cam_dir.is_dir():
                continue
            flat_name = f"{scene_dir.name}_{cam_dir.name}"
            flat_dir = DATA_RAW / flat_name
            if not flat_dir.exists():
                flat_dir.symlink_to(cam_dir)

cam_pattern = regex.compile(r"^S\d{2}_c\d{3}$")
cams = sorted(path.name for path in DATA_RAW.iterdir() if path.is_dir() and cam_pattern.match(path.name))
print(f"CityFlowV2 ready: {len(cams)} cameras")
for cam in cams:
    print(f"  {cam}")

## 4. Load 10a Stage 0/1 Checkpoint

In [ ]:
from pathlib import Path
import subprocess

PREV_OWNER_SLUG = "yahiaakhalafallah/mtmc-10a-stages-0-2"
PREV_SLUG = PREV_OWNER_SLUG.split("/", 1)[1]
INPUT_ROOT = Path("/kaggle/input")


def find_input_dir(slug: str, owner_slug: str, hints=()) -> Path:
    direct = INPUT_ROOT / slug
    if direct.exists():
        return direct

    owner, _, kernel = owner_slug.partition("/")
    nested = INPUT_ROOT / "notebooks" / owner / kernel
    if nested.exists():
        return nested

    lowered_slug = slug.lower()
    lowered_hints = tuple(str(hint).lower() for hint in hints)
    direct_children = list(INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []
    for path in direct_children:
        if not path.is_dir():
            continue
        name = path.name.lower()
        if lowered_slug in name or all(hint in name for hint in lowered_hints):
            return path

    for path in INPUT_ROOT.rglob("checkpoint.tar.gz") if INPUT_ROOT.exists() else []:
        parent_text = str(path.parent).lower()
        if lowered_slug in parent_text or all(hint in parent_text for hint in lowered_hints):
            return path.parent

    return direct


def find_checkpoint() -> Path:
    prev_input = find_input_dir(PREV_SLUG, PREV_OWNER_SLUG, hints=("10a", "stages", "0", "2"))
    checkpoint_path = prev_input / "checkpoint.tar.gz"
    if checkpoint_path.exists():
        print(f"10a input: {prev_input}")
        return checkpoint_path

    print(f"checkpoint.tar.gz not found at {checkpoint_path}; trying Kaggle API fallback")
    dl_dir = Path("/tmp/kaggle_10a_download")
    dl_dir.mkdir(parents=True, exist_ok=True)
    for candidate in [PREV_OWNER_SLUG, "gumfreddy/mtmc-10a-stages-0-2"]:
        result = subprocess.run(
            [
                "kaggle",
                "kernels",
                "output",
                candidate,
                "--file-pattern",
                r"^checkpoint\.tar\.gz$",
                "-p",
                str(dl_dir),
            ],
            capture_output=True,
            text=True,
        )
        print(result.stdout)
        print(result.stderr)
        checkpoint_path = dl_dir / "checkpoint.tar.gz"
        if checkpoint_path.exists() and checkpoint_path.stat().st_size > 0:
            print(f"10a checkpoint downloaded from {candidate}")
            return checkpoint_path

    searched = [str(path) for path in INPUT_ROOT.rglob("checkpoint.tar.gz")] if INPUT_ROOT.exists() else []
    raise FileNotFoundError(
        f"10a checkpoint.tar.gz not found for {PREV_OWNER_SLUG}; searched mount candidates and API fallback. "
        f"Visible checkpoints under /kaggle/input: {searched[:20]}"
    )


checkpoint = find_checkpoint()

EXTRACT_DIR = Path("/tmp/10a_checkpoint")
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Extracting {checkpoint} ({checkpoint.stat().st_size / 1024**2:.1f} MB)")
with tarfile.open(str(checkpoint), "r:gz") as tar:
    tar.extractall(str(EXTRACT_DIR))

with open(EXTRACT_DIR / "run_metadata.json") as handle:
    previous_meta = json.load(handle)
PREV_RUN_NAME = previous_meta["run_name"]
PREV_RUN_DIR = EXTRACT_DIR / PREV_RUN_NAME
print(f"Loaded 10a run: {PREV_RUN_NAME}")
for stage in ["stage1", "stage2"]:
    stage_dir = PREV_RUN_DIR / stage
    print(f"  {stage}: exists={stage_dir.exists()} files={len([p for p in stage_dir.rglob('*') if p.is_file()]) if stage_dir.exists() else 0}")

## 5. Run Vanilla TTA Stage 2 Only

In [ ]:
from datetime import datetime

DATA_OUT = Path("/tmp/pipeline_outputs")
DATA_OUT.mkdir(parents=True, exist_ok=True)
RUN_NAME = f"run_14c_tta_v1_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR = DATA_OUT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
shutil.copytree(PREV_RUN_DIR / "stage1", RUN_DIR / "stage1")
print(f"Run  : {RUN_NAME}")
print(f"Stage1 copied from {PREV_RUN_DIR / 'stage1'}")

In [ ]:
os.chdir(str(PROJECT))

from src.core.config import load_config, save_config
from src.core.io_utils import load_tracklets_by_camera
from src.core.logging_utils import setup_logging
from src.stage2_features import run_stage2

BASELINE_WEIGHTS = "models/reid/transreid_cityflowv2_best.pth"
TERTIARY_SLUG = "09s-dinov2-large-cityflowv2"
TERTIARY_OWNER_SLUG = f"yahiaakhalafallah/{TERTIARY_SLUG}"
TERTIARY_FILENAME = "vehicle_transreid_dinov2_large_cityflowv2_final.pth"


def find_tertiary_weights() -> str:
    candidates = [
        Path("/kaggle/input") / TERTIARY_SLUG / TERTIARY_FILENAME,
        Path("/kaggle/input/notebooks/yahiaakhalafallah") / TERTIARY_SLUG / TERTIARY_FILENAME,
    ]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for candidate in input_root.rglob(TERTIARY_FILENAME):
            if TERTIARY_SLUG in str(candidate) or "dinov2" in str(candidate).lower():
                return str(candidate)

    dl_dir = Path("/tmp/kaggle_09s_download")
    dl_dir.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(
        [
            "kaggle",
            "kernels",
            "output",
            TERTIARY_OWNER_SLUG,
            "--file-pattern",
            f"^{TERTIARY_FILENAME}$",
            "-p",
            str(dl_dir),
        ],
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    print(result.stderr)
    downloaded = dl_dir / TERTIARY_FILENAME
    if downloaded.exists() and downloaded.stat().st_size > 0:
        return str(downloaded)

    return str(candidates[0])


TERTIARY_WEIGHTS = find_tertiary_weights()
print(f"Primary TransReID: {BASELINE_WEIGHTS} exists={Path(BASELINE_WEIGHTS).exists()}")
print(f"DINOv2 tertiary : {TERTIARY_WEIGHTS} exists={Path(TERTIARY_WEIGHTS).exists()}")

overrides = [
    f"project.run_name={RUN_NAME}",
    f"project.output_dir={DATA_OUT}",
    f"stage0.input_dir={DATA_RAW}",
    "stage0.cameras=[S01_c001,S01_c002,S01_c003,S02_c006,S02_c007,S02_c008]",
    "stage2.crop.samples_per_tracklet=48",
    "stage2.crop.foreground_masking.enabled=false",
    "stage2.reid.normalize_views=true",
    "stage2.pca.n_components=384",
    "stage2.reid.color_augment=false",
    "stage2.reid.vehicle.concat_patch=false",
    "stage2.reid.vehicle.input_size=[256,256]",
    "stage2.reid.vehicle2.enabled=false",
    "stage2.multi_query.k=0",
    "stage2.camera_bn.enabled=false",
    "stage2.camera_tta.enabled=false",
    "stage2.power_norm.alpha=0.0",
]

if Path(TERTIARY_WEIGHTS).exists():
    overrides += [
        "stage2.reid.vehicle3.enabled=true",
        f"stage2.reid.vehicle3.weights_path={TERTIARY_WEIGHTS}",
        "stage2.reid.vehicle3.input_size=[252,252]",
        "stage2.reid.vehicle3.embedding_dim=1024",
        "stage2.reid.vehicle3.model_name=transreid",
        "stage2.reid.vehicle3.vit_model=vit_large_patch14_dinov2.lvd142m",
        "stage2.reid.vehicle3.clip_normalization=false",
        "stage2.reid.vehicle3.num_cameras=59",
        "stage2.reid.vehicle3.save_separate=true",
    ]
else:
    print("WARNING: DINOv2 tertiary checkpoint not found; vehicle3 will stay disabled")

cfg = load_config("configs/default.yaml", dataset_config="configs/datasets/cityflowv2.yaml", overrides=overrides)
setup_logging(level=cfg.project.get("log_level", "INFO"), log_file=RUN_DIR / "pipeline.log")
save_config(cfg, RUN_DIR / "config.yaml")
video_paths = {}
for cam_dir in sorted(DATA_RAW.glob("S*_c*")):
    if not cam_dir.is_dir():
        continue
    cam_id = cam_dir.name
    video_path = cam_dir / "vdo.avi"
    if video_path.exists():
        video_paths[cam_id] = str(video_path)
tracklets_by_camera = load_tracklets_by_camera(RUN_DIR / "stage1")
print(f"Tracklets: {sum(len(v) for v in tracklets_by_camera.values())} across {len(tracklets_by_camera)} cameras")
print(f"Videos   : {len(video_paths)} discovered")
for camera_id in sorted(tracklets_by_camera):
    print(f"  {camera_id}: tracklets={len(tracklets_by_camera[camera_id])} video={camera_id in video_paths}")

start = time.time()
features = run_stage2(
    cfg,
    tracklets_by_camera,
    video_paths,
    output_dir=RUN_DIR / "stage2",
    smoke_test=False,
    stage0_dir=None,
)
elapsed = time.time() - start
print(f"TTA Stage 2 complete: {len(features)} features in {elapsed / 60:.1f} min")

## 6. Run Stages 3-5

In [ ]:
from src.stage3_indexing import run_stage3
from src.stage4_association import run_stage4
from src.stage5_evaluation import run_stage5

AQE_K = 3
SIM_THRESH = 0.50
SOLVER = "cc"
ALGORITHM = "conflict_free_cc"
LOUVAIN_RES = 0.70

APPEARANCE_WEIGHT = 0.70
HSV_WEIGHT = 0.0
ST_WEIGHT = round(1.0 - APPEARANCE_WEIGHT - HSV_WEIGHT, 4)

BRIDGE_PRUNE = 0.0
MAX_COMP_SIZE = 12
FIC_REG = 0.50
GALLERY_THRESH = 0.48
ORPHAN_MATCH_THRESH = 0.38

INTRA_MERGE = True
INTRA_MERGE_THRESH = 0.80
INTRA_MERGE_GAP = 30

W_TERTIARY = 0.60
W_PRIMARY = round(1.0 - W_TERTIARY, 6)

MULTI_QUERY_WEIGHT = 0.00
CAMERA_BIAS = False
CAMERA_BIAS_ITERS = 2
ZONE_MODEL = False
ZONE_BONUS = 0.06
ZONE_PENALTY = 0.04
HIERARCHICAL = False
HIER_CENTROID_TH = 0.45
HIER_MERGE_TH = 0.45
HIER_ORPHAN_TH = 0.40
RERANKING = False
RERANKING_K1 = 20
RERANKING_K2 = 6
RERANKING_LAMBDA = 0.5
CAMERA_PAIR_NORM = False
AFLINK_ENABLED = False
AFLINK_MAX_TIME_GAP_FRAMES = 150
AFLINK_MAX_SPATIAL_GAP_PX = 150.0
AFLINK_MIN_DIRECTION_COS = 0.85
AFLINK_MIN_VELOCITY_RATIO = 0.5
AFLINK_VELOCITY_WINDOW = 5
MTMC_ONLY = False

PRIMARY_EMBEDDINGS_PATH = RUN_DIR / "stage2" / "embeddings.npy"
TERTIARY_DINOV2_PATH = RUN_DIR / "stage2" / "embeddings_tertiary.npy"
assert PRIMARY_EMBEDDINGS_PATH.exists(), PRIMARY_EMBEDDINGS_PATH
assert TERTIARY_DINOV2_PATH.exists(), (
    f"Missing DINOv2 tertiary embeddings at {TERTIARY_DINOV2_PATH}. "
    "This run must evaluate the production TransReID+DINOv2 score-fusion control."
)

GT_DIR = DATA_RAW
if not any((GT_DIR / cam / "gt" / "gt.txt").exists() for cam in ["S01_c001", "S01_c002", "S01_c003", "S02_c006", "S02_c007", "S02_c008"]):
    dataset_gt = Path("/kaggle/input/data-aicity-2023-track-2")
    extracted_gt = EXTRACT_DIR / "gt_annotations"
    if dataset_gt.exists():
        GT_DIR = dataset_gt
    elif extracted_gt.exists():
        GT_DIR = extracted_gt
    else:
        raise FileNotFoundError("CityFlowV2 ground-truth annotations not found")

stage345_overrides = [
    f"project.run_name={RUN_NAME}",
    f"project.output_dir={DATA_OUT}",
    f"stage0.input_dir={DATA_RAW}",
    "stage0.cameras=[S01_c001,S01_c002,S01_c003,S02_c006,S02_c007,S02_c008]",
    f"stage4.association.query_expansion.k={AQE_K}",
    "stage4.association.query_expansion.alpha=5.0",
    "stage4.association.query_expansion.dba=false",
    f"stage4.association.graph.similarity_threshold={SIM_THRESH}",
    f"stage4.association.solver={SOLVER}",
    f"stage4.association.graph.algorithm={ALGORITHM}",
    f"stage4.association.graph.louvain_resolution={LOUVAIN_RES}",
    f"stage4.association.graph.bridge_prune_margin={BRIDGE_PRUNE}",
    f"stage4.association.graph.max_component_size={MAX_COMP_SIZE}",
    f"stage4.association.weights.vehicle.appearance={APPEARANCE_WEIGHT}",
    f"stage4.association.weights.vehicle.hsv={HSV_WEIGHT}",
    f"stage4.association.weights.vehicle.spatiotemporal={ST_WEIGHT}",
    "stage4.association.mutual_nn.top_k_per_query=20",
    "stage4.association.fic.enabled=true",
    f"stage4.association.fic.regularisation={FIC_REG}",
    f"stage4.association.reranking.enabled={str(RERANKING).lower()}",
    f"stage4.association.reranking.k1={RERANKING_K1}",
    f"stage4.association.reranking.k2={RERANKING_K2}",
    f"stage4.association.reranking.lambda_value={RERANKING_LAMBDA}",
    f"stage4.association.camera_pair_norm.enabled={str(CAMERA_PAIR_NORM).lower()}",
    "stage4.association.fac.enabled=false",
    f"stage4.association.multi_query.enabled={str(MULTI_QUERY_WEIGHT > 0.0).lower()}",
    f"stage4.association.multi_query.weight={MULTI_QUERY_WEIGHT}",
    f"stage4.association.aflink.enabled={str(AFLINK_ENABLED).lower()}",
    f"stage4.association.aflink.max_time_gap_frames={AFLINK_MAX_TIME_GAP_FRAMES}",
    f"stage4.association.aflink.max_spatial_gap_px={AFLINK_MAX_SPATIAL_GAP_PX}",
    f"stage4.association.aflink.min_direction_cos={AFLINK_MIN_DIRECTION_COS}",
    f"stage4.association.aflink.min_velocity_ratio={AFLINK_MIN_VELOCITY_RATIO}",
    f"stage4.association.aflink.velocity_window={AFLINK_VELOCITY_WINDOW}",
    "stage4.association.secondary_embeddings.path=",
    "stage4.association.secondary_embeddings.weight=0.0",
    f"stage4.association.tertiary_embeddings.path={TERTIARY_DINOV2_PATH}",
    f"stage4.association.tertiary_embeddings.weight={W_TERTIARY}",
    f"stage4.association.camera_bias.enabled={str(CAMERA_BIAS).lower()}",
    f"stage4.association.camera_bias.iterations={CAMERA_BIAS_ITERS}",
    f"stage4.association.zone_model.enabled={str(ZONE_MODEL).lower()}",
    "stage4.association.zone_model.zone_data_path=configs/datasets/cityflowv2_zones.json",
    f"stage4.association.zone_model.bonus={ZONE_BONUS}",
    f"stage4.association.zone_model.penalty={ZONE_PENALTY}",
    f"stage4.association.hierarchical.enabled={str(HIERARCHICAL).lower()}",
    f"stage4.association.hierarchical.centroid_threshold={HIER_CENTROID_TH}",
    f"stage4.association.hierarchical.merge_threshold={HIER_MERGE_TH}",
    f"stage4.association.hierarchical.orphan_threshold={HIER_ORPHAN_TH}",
    "stage4.association.hierarchical.max_merge_size=12",
    f"stage4.association.intra_camera_merge.enabled={str(INTRA_MERGE).lower()}",
    f"stage4.association.intra_camera_merge.threshold={INTRA_MERGE_THRESH}",
    f"stage4.association.intra_camera_merge.max_time_gap={INTRA_MERGE_GAP}",
    "stage4.association.gallery_expansion.enabled=true",
    f"stage4.association.gallery_expansion.threshold={GALLERY_THRESH}",
    f"stage4.association.gallery_expansion.orphan_match_threshold={ORPHAN_MATCH_THRESH}",
    "stage4.association.weights.length_weight_power=0.3",
    "stage4.association.temporal_overlap.enabled=true",
    "stage4.association.temporal_overlap.bonus=0.05",
    "stage4.association.temporal_overlap.max_mean_time=5.0",
    f"stage5.mtmc_only_submission={str(MTMC_ONLY).lower()}",
    "stage5.stationary_filter.enabled=true",
    "stage5.stationary_filter.min_displacement_px=150",
    "stage5.stationary_filter.max_mean_velocity_px=2.0",
    "stage5.min_submission_confidence=0.15",
    "stage5.cross_id_nms_iou=0.40",
    "stage5.min_trajectory_confidence=0.30",
    "stage5.min_trajectory_frames=40",
    "stage5.track_edge_trim.enabled=false",
    "stage5.track_smoothing.enabled=false",
    "stage5.gt_frame_clip=true",
    "stage5.gt_zone_filter=true",
    f"stage5.ground_truth_dir={GT_DIR}",
]

cfg = load_config("configs/default.yaml", dataset_config="configs/datasets/cityflowv2.yaml", overrides=stage345_overrides)
save_config(cfg, RUN_DIR / "config.yaml")

print("Locked downstream config:")
print(f"  Stage 3: FAISS {cfg.stage3.faiss.index_type}")
print(f"  Stage 4: AQE_K={AQE_K}, sim_thresh={SIM_THRESH}, algorithm={ALGORITHM}")
print(f"  weights: primary={W_PRIMARY:.2f}, tertiary_dinov2={W_TERTIARY:.2f}, secondary=0.00")
print(f"  FIC_REG={FIC_REG}, gallery={GALLERY_THRESH}, orphan={ORPHAN_MATCH_THRESH}")
print(f"  Stage 5 GT: {GT_DIR}")

start = time.time()
faiss_index, metadata_store = run_stage3(
    cfg,
    features,
    tracklets_by_camera,
    output_dir=RUN_DIR / "stage3",
)
stage3_min = (time.time() - start) / 60.0
print(f"Stage 3 complete in {stage3_min:.2f} min")

start = time.time()
trajectories = run_stage4(
    cfg,
    faiss_index,
    metadata_store,
    features,
    tracklets_by_camera,
    output_dir=RUN_DIR / "stage4",
)
stage4_min = (time.time() - start) / 60.0
print(f"Stage 4 complete in {stage4_min:.2f} min: {len(trajectories)} global trajectories")

start = time.time()
evaluation_result = run_stage5(
    cfg,
    trajectories,
    output_dir=RUN_DIR / "stage5",
)
stage5_min = (time.time() - start) / 60.0
print(f"Stage 5 complete in {stage5_min:.2f} min")

## 7. Save IDF1 Summary

In [ ]:
def load_metrics(report_path: Path) -> dict:
    if not report_path.exists():
        return {}
    payload = json.loads(report_path.read_text(encoding="utf-8"))
    details = payload.get("details", {}) or {}
    error_analysis = details.get("error_analysis", {}) or {}
    return {
        "mtmc_idf1": payload.get("mtmc_idf1", details.get("mtmc_idf1", payload.get("idf1"))),
        "trackeval_idf1": payload.get("idf1"),
        "mota": payload.get("mota"),
        "hota": payload.get("hota"),
        "id_switches": payload.get("id_switches"),
        "conflations": error_analysis.get("conflated_pred"),
        "fragmentations": error_analysis.get("fragmented_gt"),
        "num_pred_ids": payload.get("num_pred_ids", error_analysis.get("total_pred")),
    }

report_path = RUN_DIR / "stage5" / "evaluation_report.json"
metrics = load_metrics(report_path)
idf1_value = metrics.get("mtmc_idf1")
if idf1_value is None:
    idf1_value = metrics.get("trackeval_idf1")
if idf1_value is None:
    raise RuntimeError(f"IDF1 not found in {report_path}")

checkpoint_path = Path("/kaggle/working/checkpoint.tar.gz")
metadata_path = Path("/kaggle/working/run_metadata.json")
summary_path = Path("/kaggle/working/14c_summary.json")

summary = {
    "run_name": RUN_NAME,
    "source_10a_run_name": PREV_RUN_NAME,
    "experiment": "14c-tta-stage2",
    "variant": TTA_RECIPE["variant"],
    "kernel": "yahiaakhalafallah/14c-tta-stage2",
    "tta_recipe": TTA_RECIPE,
    "idf1": idf1_value,
    "mtmc_idf1": metrics.get("mtmc_idf1"),
    "trackeval_idf1": metrics.get("trackeval_idf1"),
    "mota": metrics.get("mota"),
    "hota": metrics.get("hota"),
    "id_switches": metrics.get("id_switches"),
    "conflations": metrics.get("conflations"),
    "fragmentations": metrics.get("fragmentations"),
    "num_pred_ids": metrics.get("num_pred_ids"),
    "frame_id_convention": "0-based internal Stage 1/2 frame IDs; MOT output is converted to 1-based in Stage 5",
    "stage2_outputs": {
        "features_in_memory": len(features),
        "primary_embeddings": str(PRIMARY_EMBEDDINGS_PATH),
        "tertiary_embeddings": str(TERTIARY_DINOV2_PATH),
    },
    "fusion_config": {
        "w_primary": W_PRIMARY,
        "w_tertiary": W_TERTIARY,
        "w_secondary": 0.0,
        "aqe_k": AQE_K,
        "fic_regularisation": FIC_REG,
        "pca_components": 384,
        "similarity_threshold": SIM_THRESH,
        "algorithm": ALGORITHM,
    },
    "stage_timings_min": {
        "stage2": round(elapsed / 60.0, 2),
        "stage3": round(stage3_min, 2),
        "stage4": round(stage4_min, 2),
        "stage5": round(stage5_min, 2),
    },
    "paths": {
        "run_dir": str(RUN_DIR),
        "evaluation_report": str(report_path),
        "stage3": str(RUN_DIR / "stage3"),
        "stage4": str(RUN_DIR / "stage4"),
        "stage5": str(RUN_DIR / "stage5"),
    },
}

metadata_path.write_text(json.dumps({"run_name": RUN_NAME}, indent=2), encoding="utf-8")
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(f"Building checkpoint for {RUN_NAME}")
with tarfile.open(str(checkpoint_path), "w:gz") as tar:
    tar.add(str(metadata_path), arcname="run_metadata.json")
    tar.add(str(summary_path), arcname="14c_summary.json")
    for stage in ["stage1", "stage2", "stage3", "stage4", "stage5"]:
        stage_dir = RUN_DIR / stage
        count = 0
        if stage_dir.exists():
            for file_path in stage_dir.rglob("*"):
                if file_path.is_file():
                    tar.add(str(file_path), arcname=f"{RUN_NAME}/{stage}/{file_path.relative_to(stage_dir)}")
                    count += 1
        print(f"  added {stage}: {count} files")

    gt_root = EXTRACT_DIR / "gt_annotations"
    gt_count = 0
    if gt_root.exists():
        for file_path in gt_root.rglob("*"):
            if file_path.is_file():
                tar.add(str(file_path), arcname=f"gt_annotations/{file_path.relative_to(gt_root)}")
                gt_count += 1
    print(f"  added gt_annotations: {gt_count} files")

size_mb = checkpoint_path.stat().st_size / 1024**2
print(f"Saved summary: {summary_path}")
print(f"Checkpoint: {checkpoint_path} ({size_mb:.1f} MB)")
print(json.dumps(summary, indent=2))